# 02 - Data Cleaning

## Tujuan

Notebook ini digunakan untuk membersihkan dan memvalidasi
dataset Customer Segmentation berdasarkan masalah yang ditemukan
pada tahap Data Understanding.

Proses cleaning mencakup:

- memperbaiki tipe data
- menangani duplicate
- menangani missing values
- menstandarkan categorical values
- melakukan validasi numerical values
- memeriksa outlier
- melakukan relationship validation
- melakukan final data quality validation

Raw dataset tidak dimodifikasi secara langsung.
Hasil cleaning akan disimpan pada folder `data/processed/`.

# 02 - Data Cleaning

In [12]:
import pandas as pd
import numpy as np

In [13]:
customers = pd.read_csv(
    "../data/raw/customers.csv"
)

transactions = pd.read_csv(
    "../data/raw/transactions.csv"
)

In [14]:
customers.head()

,customer_id,customer_name,gender,city,registration_date,province
0,CUST00001,Customer 0001,Male,Jakarta,2025-06-01,DKI Jakarta
1,CUST00002,Customer 0002,Female,Bekasi,2023-11-09,Jawa Barat
2,CUST00003,Customer 0003,Male,Yogyakarta,2025-04-19,DI Yogyakarta
3,CUST00004,Customer 0004,Male,Depok,2024-06-02,Jawa Barat
4,CUST00005,Customer 0005,Male,Semarang,2024-01-10,Jawa Tengah


In [15]:
transactions.head()

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method
0,TRX0000001,CUST01170,2025-10-14,Beauty,1,241000,10,216900,E-Wallet
1,TRX0000002,CUST00686,2025-05-15,Beauty,1,37000,10,33300,COD
2,TRX0000003,CUST01245,2024-11-29,Fashion,2,74000,30,103600,Credit Card
3,TRX0000004,CUST00959,2024-11-09,Fashion,3,59000,30,123900,E-Wallet
4,TRX0000005,CUST00478,2025-06-19,Fashion,2,151000,10,271800,Credit Card


In [16]:
customers_raw_rows = len(customers)
transactions_raw_rows = len(transactions)

print(f"Customers raw rows    : {customers_raw_rows:,}")
print(f"Transactions raw rows : {transactions_raw_rows:,}")

Customers raw rows    : 2,000
Transactions raw rows : 25,025


Perbaiki Data Type

In [20]:
customers_clean = customers
transactions_clean = transactions

customers_clean["registration_date"] = pd.to_datetime(
    customers_clean["registration_date"],
    errors="coerce"
)

transactions_clean["transaction_date"] = pd.to_datetime(
    transactions_clean["transaction_date"],
    errors="coerce"
)

Validasi conversion

In [21]:
print(
    customers_clean["registration_date"]
    .isna()
    .sum()
)

0


In [22]:
print(
    transactions_clean["transaction_date"]
    .isna()
    .sum()
)

0


Periksa tipe data:

In [23]:
customers_clean.dtypes

customer_id                  object
customer_name                object
gender                       object
city                         object
registration_date    datetime64[ns]
province                     object
dtype: object

In [24]:
transactions_clean.dtypes

transaction_id              object
customer_id                 object
transaction_date    datetime64[ns]
category                    object
quantity                     int64
unit_price                   int64
discount_pct                 int64
revenue                      int64
payment_method              object
dtype: object

Tangani Exact Duplicate

In [25]:
duplicate_rows = (
    transactions_clean
    .duplicated()
    .sum()
)

print(
    f"Exact duplicate rows: {duplicate_rows}"
)

Exact duplicate rows: 25


In [26]:
transactions_clean[
    transactions_clean.duplicated(
        keep=False
    )
].sort_values("transaction_id")

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method
2227,TRX0002228,CUST01472,2024-03-14,Home & Living,1,368000,15,312800,COD
25023,TRX0002228,CUST01472,2024-03-14,Home & Living,1,368000,15,312800,COD
2518,TRX0002519,CUST00933,2024-03-26,Beauty,1,211000,30,147700,Bank Transfer
25014,TRX0002519,CUST00933,2024-03-26,Beauty,1,211000,30,147700,Bank Transfer
3386,TRX0003387,CUST01815,2024-01-26,Food & Beverage,1,19000,20,15200,Credit Card
25008,TRX0003387,CUST01815,2024-01-26,Food & Beverage,1,19000,20,15200,Credit Card
3503,TRX0003504,CUST01790,2025-08-07,Beauty,4,216000,25,648000,Credit Card
25011,TRX0003504,CUST01790,2025-08-07,Beauty,4,216000,25,648000,Credit Card
5804,TRX0005805,CUST01268,2024-06-15,Fashion,1,63000,10,56700,E-Wallet
25006,TRX0005805,CUST01268,2024-06-15,Fashion,1,63000,10,56700,E-Wallet


Hapus Exact Duplicate

In [27]:
transactions_clean = (
    transactions_clean
    .drop_duplicates()
    .copy()
)

In [28]:
transactions_clean.duplicated().sum()

np.int64(0)

In [29]:
transactions_clean[
    transactions_clean[
        "transaction_id"
    ].duplicated(keep=False)
]

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method


In [30]:
transactions_clean[
    "transaction_id"
].duplicated().sum()

np.int64(0)

Missing Values — Customers

In [32]:
customers_clean.isna().sum()

customer_id          0
customer_name        0
gender               8
city                 0
registration_date    0
province             0
dtype: int64

In [33]:
customers_clean["gender"] = (
    customers_clean["gender"]
    .fillna("Unknown")
)

Missing Province

In [34]:
customers_clean[
    customers_clean["province"].isna()
]

,customer_id,customer_name,gender,city,registration_date,province


In [35]:
customers_clean[
    "city"
].value_counts(dropna=False)

city
Jakarta       547
Bandung       265
Surabaya      239
Tangerang     199
Bekasi        167
Depok         141
Semarang      124
Makassar      115
Yogyakarta    108
Medan          79
BANDUNG         8
 jakarta        8
Name: count, dtype: int64

In [36]:
customers_clean["city"] = (
    customers_clean["city"]
    .astype("string")
    .str.strip()
    .str.title()
)

Validasi City

In [37]:
customers_clean[
    "city"
].value_counts(dropna=False)

city
Jakarta       555
Bandung       273
Surabaya      239
Tangerang     199
Bekasi        167
Depok         141
Semarang      124
Makassar      115
Yogyakarta    108
Medan          79
Name: count, dtype: Int64

In [38]:
customers_clean[
    "city"
].nunique()

10

Isi Province berdasarkan City

In [39]:
province_map = {
    "Jakarta": "DKI Jakarta",
    "Bandung": "Jawa Barat",
    "Surabaya": "Jawa Timur",
    "Bekasi": "Jawa Barat",
    "Tangerang": "Banten",
    "Depok": "Jawa Barat",
    "Semarang": "Jawa Tengah",
    "Yogyakarta": "DI Yogyakarta",
    "Medan": "Sumatera Utara",
    "Makassar": "Sulawesi Selatan"
}

In [40]:
customers_clean["province"] = (
    customers_clean["province"]
    .fillna(
        customers_clean["city"]
        .map(province_map)
    )
)

Validasi Province

In [41]:
customers_clean[
    "province"
].isna().sum()

np.int64(0)

In [42]:
customers_clean[
    ["city", "province"]
].drop_duplicates().sort_values("city")

,city,province
21,Bandung,Jawa Barat
238,Bandung,Sulawesi Selatan
1052,Bandung,DKI Jakarta
1474,Bandung,Jawa Timur
1,Bekasi,Jawa Barat
3,Depok,Jawa Barat
0,Jakarta,DKI Jakarta
287,Jakarta,Jawa Barat
718,Jakarta,Sulawesi Selatan
927,Jakarta,Jawa Timur


In [43]:
customers_clean[
    customers_clean["province"].isna()
][
    ["customer_id", "city", "province"]
]

,customer_id,city,province


In [44]:
customers_clean["province"].isna().sum()

np.int64(0)

In [45]:
customers_clean.loc[
    customers_clean["province"].isna(),
    "city"
].value_counts(dropna=False)

Series([], Name: count, dtype: Int64)

In [46]:
customers_clean["city"] = (
    customers_clean["city"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [47]:
customers_clean["city"].value_counts(dropna=False)

city
Jakarta       555
Bandung       273
Surabaya      239
Tangerang     199
Bekasi        167
Depok         141
Semarang      124
Makassar      115
Yogyakarta    108
Medan          79
Name: count, dtype: Int64

In [48]:
customers_clean[
    ["city", "province"]
].drop_duplicates().sort_values("city")

,city,province
21,Bandung,Jawa Barat
238,Bandung,Sulawesi Selatan
1052,Bandung,DKI Jakarta
1474,Bandung,Jawa Timur
1,Bekasi,Jawa Barat
3,Depok,Jawa Barat
0,Jakarta,DKI Jakarta
287,Jakarta,Jawa Barat
718,Jakarta,Sulawesi Selatan
927,Jakarta,Jawa Timur


Mengukur Missing Value

In [50]:
expected_province = (
    customers_clean["city"]
    .map(province_map)
)

province_mismatch = (
    customers_clean["province"].notna()
    & expected_province.notna()
    & (customers_clean["province"] != expected_province)
)

print(
    "Province mismatch:",
    province_mismatch.sum()
)

Province mismatch: 11


In [51]:
customers_clean.loc[
    province_mismatch,
    [
        "customer_id",
        "city",
        "province"
    ]
]

,customer_id,city,province
238,CUST00239,Bandung,Sulawesi Selatan
287,CUST00288,Jakarta,Jawa Barat
623,CUST00624,Bandung,Sulawesi Selatan
718,CUST00719,Jakarta,Sulawesi Selatan
927,CUST00928,Jakarta,Jawa Timur
1052,CUST01053,Bandung,DKI Jakarta
1162,CUST01163,Jakarta,Jawa Barat
1474,CUST01475,Bandung,Jawa Timur
1668,CUST01669,Jakarta,Jawa Barat
1736,CUST01737,Bandung,Sulawesi Selatan


In [52]:
customers_clean["province"] = (
    customers_clean["city"]
    .map(province_map)
)

Validasi

In [53]:
customers_clean[
    ["city", "province"]
].drop_duplicates().sort_values("city")

,city,province
21,Bandung,Jawa Barat
1,Bekasi,Jawa Barat
3,Depok,Jawa Barat
0,Jakarta,DKI Jakarta
11,Makassar,Sulawesi Selatan
12,Medan,Sumatera Utara
4,Semarang,Jawa Tengah
9,Surabaya,Jawa Timur
5,Tangerang,Banten
2,Yogyakarta,DI Yogyakarta


In [54]:
expected_province = (
    customers_clean["city"]
    .map(province_map)
)

province_mismatch = (
    customers_clean["province"]
    != expected_province
)

province_mismatch.sum()

np.int64(0)

In [55]:
customers_clean[
    ["city", "province"]
].isna().sum()

city        0
province    0
dtype: int64

### Standardisasi City dan Province

Pada tahap Data Quality Assessment ditemukan beberapa kombinasi
`city` dan `province` yang tidak konsisten.

Contoh:

- Bandung → Sulawesi Selatan
- Bandung → DKI Jakarta
- Jakarta → Jawa Barat
- Jakarta → Jawa Timur

Karena setiap kota pada dataset memiliki mapping provinsi yang
dapat ditentukan secara konsisten, kolom `city` terlebih dahulu
distandardisasi, kemudian `province` dibangun kembali berdasarkan
mapping city-to-province.

Setelah proses cleaning, dilakukan validasi ulang untuk memastikan
setiap kota hanya terhubung dengan provinsi yang sesuai.

Data Cleaning — Lanjutan

In [56]:
transactions_clean.head()

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method
0,TRX0000001,CUST01170,2025-10-14,Beauty,1,241000,10,216900,E-Wallet
1,TRX0000002,CUST00686,2025-05-15,Beauty,1,37000,10,33300,COD
2,TRX0000003,CUST01245,2024-11-29,Fashion,2,74000,30,103600,Credit Card
3,TRX0000004,CUST00959,2024-11-09,Fashion,3,59000,30,123900,E-Wallet
4,TRX0000005,CUST00478,2025-06-19,Fashion,2,151000,10,271800,Credit Card


In [57]:
transactions_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 0 to 24999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    25000 non-null  object        
 1   customer_id       25000 non-null  object        
 2   transaction_date  25000 non-null  datetime64[ns]
 3   category          25000 non-null  object        
 4   quantity          25000 non-null  int64         
 5   unit_price        25000 non-null  int64         
 6   discount_pct      25000 non-null  int64         
 7   revenue           25000 non-null  int64         
 8   payment_method    24970 non-null  object        
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 1.9+ MB


In [58]:
transactions_clean["category"].value_counts(dropna=False)

category
Fashion            6912
Beauty             4530
Home & Living      3821
Electronics        3727
Food & Beverage    3469
Sports             2521
fashion              20
Name: count, dtype: int64

In [59]:
transactions_clean["payment_method"].value_counts(dropna=False)

payment_method
Bank Transfer      5052
COD                4997
Virtual Account    4996
Credit Card        4969
E-Wallet           4956
NaN                  30
Name: count, dtype: int64

Standardisasi category

In [60]:
transactions_clean["category"].nunique(dropna=False)

7

In [61]:
transactions_clean["category"] = (
    transactions_clean["category"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [62]:
transactions_clean[
    "category"
].value_counts(dropna=False)

category
Fashion            6932
Beauty             4530
Home & Living      3821
Electronics        3727
Food & Beverage    3469
Sports             2521
Name: count, dtype: Int64

Missing value pada category

In [64]:
transactions_clean[
    "category"
].isna().sum()

np.int64(0)

Standardisasi payment_method

In [65]:
transactions_clean[
    "payment_method"
].value_counts(dropna=False)

payment_method
Bank Transfer      5052
COD                4997
Virtual Account    4996
Credit Card        4969
E-Wallet           4956
NaN                  30
Name: count, dtype: int64

In [66]:
transactions_clean["payment_method"] = (
    transactions_clean["payment_method"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [67]:
transactions_clean[
    "payment_method"
].value_counts(dropna=False)

payment_method
Bank Transfer      5052
Cod                4997
Virtual Account    4996
Credit Card        4969
E-Wallet           4956
<NA>                 30
Name: count, dtype: Int64

In [68]:
transactions_clean["payment_method"] = (
    transactions_clean["payment_method"]
    .fillna("Unknown")
)

In [69]:
transactions_clean[
    "payment_method"
].isna().sum()

np.int64(0)

In [70]:
transactions_clean[
    "payment_method"
].value_counts()

payment_method
Bank Transfer      5052
Cod                4997
Virtual Account    4996
Credit Card        4969
E-Wallet           4956
Unknown              30
Name: count, dtype: Int64

In [71]:
transactions_clean["payment_method"] = (
    transactions_clean["payment_method"]
    .replace({
        "Cod": "COD"
    })
)

In [72]:
transactions_clean[
    "payment_method"
].value_counts()

payment_method
Bank Transfer      5052
COD                4997
Virtual Account    4996
Credit Card        4969
E-Wallet           4956
Unknown              30
Name: count, dtype: Int64

Periksa seluruh missing values lagi

In [74]:
customers_clean.isna().sum()

customer_id          0
customer_name        0
gender               0
city                 0
registration_date    0
province             0
dtype: int64

In [75]:
def missing_report(df):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().sum()
            / len(df)
            * 100
        )
    })

    return report.sort_values(
        "missing_count",
        ascending=False
    )

In [76]:
missing_report(customers_clean)

,missing_count,missing_percentage
customer_id,0,0.0
customer_name,0,0.0
gender,0,0.0
city,0,0.0
registration_date,0,0.0
province,0,0.0


In [77]:
missing_report(transactions_clean)

,missing_count,missing_percentage
transaction_id,0,0.0
customer_id,0,0.0
transaction_date,0,0.0
category,0,0.0
quantity,0,0.0
unit_price,0,0.0
discount_pct,0,0.0
revenue,0,0.0
payment_method,0,0.0


Critical Fields

In [78]:
critical_columns = [
    "transaction_id",
    "customer_id",
    "transaction_date",
    "revenue"
]

In [79]:
transactions_clean[
    critical_columns
].isna().sum()

transaction_id      0
customer_id         0
transaction_date    0
revenue             0
dtype: int64

Numerical Data Validation

In [80]:
numeric_columns = [
    "quantity",
    "unit_price",
    "discount_pct",
    "revenue"
]

transactions_clean[
    numeric_columns
].describe()

,quantity,unit_price,discount_pct,revenue
count,25000.000000,2.500000e+04,25000.000000,2.500000e+04
mean,1.706080,3.203645e+05,12.269400,4.781016e+05
std,1.026905,4.789821e+05,10.452617,8.850785e+05
min,1.000000,1.500000e+04,0.000000,9.000000e+03
25%,1.000000,7.400000e+04,5.000000,9.180000e+04
50%,1.000000,1.590000e+05,10.000000,2.016000e+05
75%,2.000000,3.290000e+05,20.000000,4.674000e+05
max,5.000000,3.486000e+06,40.000000,1.327500e+07


Validasi quantity

In [81]:
invalid_quantity = (
    transactions_clean["quantity"] <= 0
)

invalid_quantity.sum()

np.int64(0)

In [82]:
transactions_clean[
    invalid_quantity
]

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method


Validasi unit_price

In [83]:
invalid_price = (
    transactions_clean["unit_price"] <= 0
)

invalid_price.sum()

np.int64(0)

In [84]:
transactions_clean[
    invalid_price
]

,transaction_id,customer_id,transaction_date,category,quantity,unit_price,discount_pct,revenue,payment_method


Validasi diskon

In [85]:
invalid_discount = (
    ~transactions_clean[
        "discount_pct"
    ].between(0, 100)
)

invalid_discount.sum()

np.int64(0)

Validasi Revenue

In [86]:
invalid_revenue = (
    transactions_clean["revenue"] < 0
)

invalid_revenue.sum()

np.int64(0)

Numerical Validation Summary

In [87]:
validation_summary = pd.DataFrame({
    "rule": [
        "Quantity <= 0",
        "Unit Price <= 0",
        "Discount outside 0-100",
        "Revenue < 0"
    ],
    "invalid_records": [
        invalid_quantity.sum(),
        invalid_price.sum(),
        invalid_discount.sum(),
        invalid_revenue.sum()
    ]
})

validation_summary

,rule,invalid_records
0,Quantity <= 0,0
1,Unit Price <= 0,0
2,Discount outside 0-100,0
3,Revenue < 0,0


Validasi formula Revenue

In [88]:
transactions_clean[
    "expected_revenue"
] = (
    transactions_clean["quantity"]
    * transactions_clean["unit_price"]
    * (
        1
        - transactions_clean[
            "discount_pct"
        ] / 100
    )
)

In [89]:
transactions_clean[
    [
        "quantity",
        "unit_price",
        "discount_pct",
        "revenue",
        "expected_revenue"
    ]
].head(10)

,quantity,unit_price,discount_pct,revenue,expected_revenue
0,1,241000,10,216900,216900.0
1,1,37000,10,33300,33300.0
2,2,74000,30,103600,103600.0
3,3,59000,30,123900,123900.0
4,2,151000,10,271800,271800.0
5,2,323000,10,581400,581400.0
6,1,143000,15,121550,121550.0
7,1,54000,5,51300,51300.0
8,2,39000,5,74100,74100.0
9,2,910000,0,1820000,1820000.0


Hitung perbedaannya

In [90]:
transactions_clean[
    "revenue_difference"
] = (
    transactions_clean["revenue"]
    - transactions_clean[
        "expected_revenue"
    ]
).abs()

In [91]:
transactions_clean[
    "revenue_difference"
].describe()

count    2.500000e+04
mean     6.881601e-13
std      1.391212e-11
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.313226e-10
Name: revenue_difference, dtype: float64

In [92]:
(
    transactions_clean[
        "revenue_difference"
    ] > 1
).sum()

np.int64(0)

### Revenue Validation

Kolom `revenue` divalidasi dengan menghitung ulang nilai
berdasarkan formula:

Revenue = Quantity × Unit Price × (1 - Discount Percentage)

Hasil validasi menunjukkan bahwa nilai revenue konsisten dengan
komponen transaksi yang tersedia.

Hapus helper columns

In [93]:
transactions_clean.drop(
    columns=[
        "expected_revenue",
        "revenue_difference"
    ],
    inplace=True
)

masuk ke Outlier

In [94]:
transactions_clean[
    [
        "quantity",
        "unit_price",
        "revenue"
    ]
].describe()

,quantity,unit_price,revenue
count,25000.000000,2.500000e+04,2.500000e+04
mean,1.706080,3.203645e+05,4.781016e+05
std,1.026905,4.789821e+05,8.850785e+05
min,1.000000,1.500000e+04,9.000000e+03
25%,1.000000,7.400000e+04,9.180000e+04
50%,1.000000,1.590000e+05,2.016000e+05
75%,2.000000,3.290000e+05,4.674000e+05
max,5.000000,3.486000e+06,1.327500e+07


In [95]:
transactions_clean.nlargest(
    20,
    "unit_price"
)[
    [
        "transaction_id",
        "category",
        "quantity",
        "unit_price",
        "discount_pct",
        "revenue"
    ]
]

,transaction_id,category,quantity,unit_price,discount_pct,revenue
4472,TRX0004473,Electronics,2,3486000,0,6972000
22058,TRX0022059,Electronics,1,3485000,5,3310750
6835,TRX0006836,Electronics,1,3483000,0,3483000
8354,TRX0008355,Electronics,1,3476000,30,2433200
17507,TRX0017508,Electronics,3,3476000,10,9385200
11319,TRX0011320,Electronics,1,3475000,10,3127500
21785,TRX0021786,Electronics,3,3469000,25,7805250
864,TRX0000865,Electronics,4,3468000,5,13178400
23926,TRX0023927,Electronics,1,3462000,5,3288900
11835,TRX0011836,Electronics,1,3455000,0,3455000


Deteksi outlier menggunakan IQR

In [96]:
Q1 = transactions_clean[
    "unit_price"
].quantile(0.25)

Q3 = transactions_clean[
    "unit_price"
].quantile(0.75)

IQR = Q3 - Q1

In [97]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [98]:
print(f"Q1          : Rp{Q1:,.0f}")
print(f"Q3          : Rp{Q3:,.0f}")
print(f"IQR         : Rp{IQR:,.0f}")
print(f"Lower Bound : Rp{lower_bound:,.0f}")
print(f"Upper Bound : Rp{upper_bound:,.0f}")

Q1          : Rp74,000
Q3          : Rp329,000
IQR         : Rp255,000
Lower Bound : Rp-308,500
Upper Bound : Rp711,500


Cari outlier

In [99]:
price_outliers = transactions_clean[
    (
        transactions_clean[
            "unit_price"
        ] < lower_bound
    )
    |
    (
        transactions_clean[
            "unit_price"
        ] > upper_bound
    )
]

print(
    f"Price outliers: {len(price_outliers):,}"
)

Price outliers: 2,755


In [100]:
outlier_percentage = (
    len(price_outliers)
    / len(transactions_clean)
    * 100
)

print(
    f"Outlier percentage: "
    f"{outlier_percentage:.2f}%"
)

Outlier percentage: 11.02%


Investigasi outlier berdasarkan category

In [102]:
price_outliers.groupby(
    "category"
)[
    "unit_price"
].agg(
    ["count", "min", "median", "max"]
).sort_values(
    "count",
    ascending=False
)

,count,min,median,max
category,,,,
Electronics,1671,712000,1570000.0,3486000
Sports,551,712000,995000.0,1498000
Home & Living,532,714000,926000.0,1199000
Fashion,1,714000,714000.0,714000


Bandingkan statistik berdasarkan kategori

In [103]:
transactions_clean.groupby(
    "category"
)[
    "unit_price"
].agg(
    [
        "count",
        "min",
        "median",
        "mean",
        "max"
    ]
).sort_values(
    "median",
    ascending=False
)

,count,min,median,mean,max
category,,,,,
Electronics,3727,100000,585000.0,944517.037832,3486000
Sports,2521,50000,275000.0,423199.523998,1498000
Home & Living,3821,30000,185000.0,313284.742214,1199000
Fashion,6932,41000,159000.0,193835.978073,714000
Beauty,4530,25000,91000.0,120502.869757,349000
Food & Beverage,3469,15000,69000.0,96685.500144,299000


### Outlier Investigation

Outlier pada `unit_price` dideteksi menggunakan metode IQR.

Namun, nilai harga ekstrem tidak langsung dihapus karena dataset
memiliki beberapa kategori produk dengan rentang harga yang
berbeda secara signifikan, seperti Electronics dibandingkan
Food & Beverage.

Nilai ekstrem dipertahankan selama:

- quantity valid;
- unit price positif;
- discount berada pada rentang valid;
- revenue konsisten dengan formula transaksi.

Pendekatan ini menghindari penghapusan transaksi bernilai tinggi
yang masih masuk akal secara bisnis.

In [104]:
invalid_customer_reference = (
    ~transactions_clean[
        "customer_id"
    ].isin(
        customers_clean[
            "customer_id"
        ]
    )
)

In [105]:
invalid_customer_reference.sum()

np.int64(0)

Customer tanpa transaksi

In [106]:
customers_without_transactions = (
    ~customers_clean[
        "customer_id"
    ].isin(
        transactions_clean[
            "customer_id"
        ]
    )
)

In [107]:
customers_without_transactions.sum()

np.int64(102)

In [108]:
customers_clean[
    customers_without_transactions
][
    [
        "customer_id",
        "customer_name",
        "city",
        "registration_date"
    ]
].head()

,customer_id,customer_name,city,registration_date
2,CUST00003,Customer 0003,Yogyakarta,2025-04-19
10,CUST00011,Customer 0011,Jakarta,2023-04-23
72,CUST00073,Customer 0073,Jakarta,2024-07-13
120,CUST00121,Customer 0121,Yogyakarta,2023-02-08
128,CUST00129,Customer 0129,Bandung,2023-10-26


Validasi Registration Date vs Transaction Date

In [109]:
transaction_registration_check = (
    transactions_clean[
        [
            "transaction_id",
            "customer_id",
            "transaction_date"
        ]
    ]
    .merge(
        customers_clean[
            [
                "customer_id",
                "registration_date"
            ]
        ],
        on="customer_id",
        how="left"
    )
)

In [110]:
invalid_registration_date = (
    transaction_registration_check[
        "transaction_date"
    ]
    <
    transaction_registration_check[
        "registration_date"
    ]
)

In [111]:
invalid_registration_date.sum()

np.int64(8172)

In [112]:
transaction_registration_check[
    invalid_registration_date
].head(20)

,transaction_id,customer_id,transaction_date,registration_date
5,TRX0000006,CUST00564,2024-07-10,2025-04-04
7,TRX0000008,CUST00358,2024-09-27,2025-06-16
8,TRX0000009,CUST00599,2024-12-02,2025-12-08
13,TRX0000014,CUST01914,2024-07-13,2024-08-08
20,TRX0000021,CUST01047,2024-12-26,2025-02-22
21,TRX0000022,CUST01248,2024-02-16,2025-05-04
24,TRX0000025,CUST00562,2024-04-08,2024-04-09
31,TRX0000032,CUST00074,2024-12-01,2025-01-14
34,TRX0000035,CUST01869,2025-03-25,2025-07-05
37,TRX0000038,CUST01889,2024-05-09,2024-09-01


### Registration Date Validation

Ditemukan beberapa transaksi yang memiliki `transaction_date`
lebih awal dibandingkan `registration_date`.

Ketidaksesuaian ini berasal dari proses pembuatan data sintetis,
di mana tanggal registrasi dan transaksi dihasilkan secara
independen.

Karena `registration_date` tidak digunakan dalam perhitungan
utama RFM, record transaksi tetap dipertahankan.

Keterbatasan ini didokumentasikan dan perlu diperbaiki apabila
dataset digunakan untuk analisis customer tenure atau lifecycle.

Validasi Primary Key Customer

In [113]:
customer_id_duplicates = (
    customers_clean["customer_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate customer_id: "
    f"{customer_id_duplicates}"
)

Duplicate customer_id: 0


In [114]:
assert (
    customers_clean["customer_id"]
    .duplicated()
    .sum()
    == 0
)

Validasi Primary Key Transaction

In [115]:
transaction_id_duplicates = (
    transactions_clean["transaction_id"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate transaction_id: "
    f"{transaction_id_duplicates}"
)

Duplicate transaction_id: 0


In [116]:
assert (
    transactions_clean["transaction_id"]
    .duplicated()
    .sum()
    == 0
)

Validasi Foreign Key

In [117]:
invalid_customer_ids = (
    ~transactions_clean["customer_id"]
    .isin(customers_clean["customer_id"])
)

print(
    "Invalid customer references:",
    invalid_customer_ids.sum()
)

Invalid customer references: 0


In [118]:
assert invalid_customer_ids.sum() == 0

Periksa Data Type Final

In [119]:
customers_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        2000 non-null   object        
 1   customer_name      2000 non-null   object        
 2   gender             2000 non-null   object        
 3   city               2000 non-null   string        
 4   registration_date  2000 non-null   datetime64[ns]
 5   province           2000 non-null   object        
dtypes: datetime64[ns](1), object(4), string(1)
memory usage: 93.9+ KB


In [120]:
transactions_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25000 entries, 0 to 24999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    25000 non-null  object        
 1   customer_id       25000 non-null  object        
 2   transaction_date  25000 non-null  datetime64[ns]
 3   category          25000 non-null  string        
 4   quantity          25000 non-null  int64         
 5   unit_price        25000 non-null  int64         
 6   discount_pct      25000 non-null  int64         
 7   revenue           25000 non-null  int64         
 8   payment_method    25000 non-null  string        
dtypes: datetime64[ns](1), int64(4), object(2), string(2)
memory usage: 1.9+ MB


Final Missing Value Check

In [121]:
print("CUSTOMERS")
print(
    customers_clean
    .isna()
    .sum()
)

print("\nTRANSACTIONS")
print(
    transactions_clean
    .isna()
    .sum()
)

CUSTOMERS
customer_id          0
customer_name        0
gender               0
city                 0
registration_date    0
province             0
dtype: int64

TRANSACTIONS
transaction_id      0
customer_id         0
transaction_date    0
category            0
quantity            0
unit_price          0
discount_pct        0
revenue             0
payment_method      0
dtype: int64


Final Duplicate Check

In [122]:
print(
    "Customer duplicate rows:",
    customers_clean.duplicated().sum()
)

print(
    "Transaction duplicate rows:",
    transactions_clean.duplicated().sum()
)

Customer duplicate rows: 0
Transaction duplicate rows: 0


In [123]:
print(
    "Duplicate customer IDs:",
    customers_clean[
        "customer_id"
    ].duplicated().sum()
)

print(
    "Duplicate transaction IDs:",
    transactions_clean[
        "transaction_id"
    ].duplicated().sum()
)

Duplicate customer IDs: 0
Duplicate transaction IDs: 0


Final Numerical Validation

In [124]:
final_validation = {
    "duplicate_customers":
        customers_clean[
            "customer_id"
        ].duplicated().sum(),

    "duplicate_transactions":
        transactions_clean[
            "transaction_id"
        ].duplicated().sum(),

    "missing_customer_id":
        customers_clean[
            "customer_id"
        ].isna().sum(),

    "missing_transaction_id":
        transactions_clean[
            "transaction_id"
        ].isna().sum(),

    "missing_transaction_date":
        transactions_clean[
            "transaction_date"
        ].isna().sum(),

    "invalid_quantity":
        (
            transactions_clean[
                "quantity"
            ] <= 0
        ).sum(),

    "invalid_price":
        (
            transactions_clean[
                "unit_price"
            ] <= 0
        ).sum(),

    "invalid_discount":
        (
            ~transactions_clean[
                "discount_pct"
            ].between(0, 100)
        ).sum(),

    "negative_revenue":
        (
            transactions_clean[
                "revenue"
            ] < 0
        ).sum(),

    "invalid_customer_reference":
        invalid_customer_ids.sum()
}

In [125]:
final_validation

{'duplicate_customers': np.int64(0),
 'duplicate_transactions': np.int64(0),
 'missing_customer_id': np.int64(0),
 'missing_transaction_id': np.int64(0),
 'missing_transaction_date': np.int64(0),
 'invalid_quantity': np.int64(0),
 'invalid_price': np.int64(0),
 'invalid_discount': np.int64(0),
 'negative_revenue': np.int64(0),
 'invalid_customer_reference': np.int64(0)}

Ubah Validation menjadi DataFrame

In [126]:
validation_report = pd.DataFrame(
    final_validation.items(),
    columns=[
        "validation_rule",
        "invalid_records"
    ]
)

validation_report

,validation_rule,invalid_records
0,duplicate_customers,0
1,duplicate_transactions,0
2,missing_customer_id,0
3,missing_transaction_id,0
4,missing_transaction_date,0
5,invalid_quantity,0
6,invalid_price,0
7,invalid_discount,0
8,negative_revenue,0
9,invalid_customer_reference,0


Automated Assertion

In [127]:
assert (
    validation_report[
        "invalid_records"
    ] == 0
).all()

In [128]:
print(
    "✓ All critical data validation checks passed."
)

✓ All critical data validation checks passed.


Bandingkan Raw vs Clean

In [129]:
customers_clean_rows = len(
    customers_clean
)

customers_removed = (
    customers_raw_rows
    - customers_clean_rows
)

In [130]:
transactions_clean_rows = len(
    transactions_clean
)

transactions_removed = (
    transactions_raw_rows
    - transactions_clean_rows
)

In [131]:
print("CUSTOMERS")
print(
    f"Raw rows     : "
    f"{customers_raw_rows:,}"
)
print(
    f"Clean rows   : "
    f"{customers_clean_rows:,}"
)
print(
    f"Rows removed : "
    f"{customers_removed:,}"
)

print("\nTRANSACTIONS")
print(
    f"Raw rows     : "
    f"{transactions_raw_rows:,}"
)
print(
    f"Clean rows   : "
    f"{transactions_clean_rows:,}"
)
print(
    f"Rows removed : "
    f"{transactions_removed:,}"
)

CUSTOMERS
Raw rows     : 2,000
Clean rows   : 2,000
Rows removed : 0

TRANSACTIONS
Raw rows     : 25,025
Clean rows   : 25,000
Rows removed : 25


Cleaning Summary Table

In [132]:
cleaning_summary = pd.DataFrame({
    "dataset": [
        "customers",
        "transactions"
    ],

    "raw_rows": [
        customers_raw_rows,
        transactions_raw_rows
    ],

    "clean_rows": [
        len(customers_clean),
        len(transactions_clean)
    ]
})

In [133]:
cleaning_summary[
    "rows_removed"
] = (
    cleaning_summary["raw_rows"]
    - cleaning_summary["clean_rows"]
)

In [134]:
cleaning_summary

,dataset,raw_rows,clean_rows,rows_removed
0,customers,2000,2000,0
1,transactions,25025,25000,25


Cleaning Log

In [135]:
cleaning_log = pd.DataFrame({
    "issue": [
        "Duplicate transactions",
        "Inconsistent city names",
        "Inconsistent province mapping",
        "Missing gender",
        "Inconsistent category",
        "Missing payment method",
        "Incorrect date type",
        "Price outliers"
    ],

    "action": [
        "Removed exact duplicates",
        "Trimmed and standardized capitalization",
        "Rebuilt province using city mapping",
        "Filled with Unknown",
        "Trimmed and standardized capitalization",
        "Filled with Unknown",
        "Converted to datetime",
        "Investigated and retained valid values"
    ]
})

In [136]:
cleaning_log

,issue,action
0,Duplicate transactions,Removed exact duplicates
1,Inconsistent city names,Trimmed and standardized capitalization
2,Inconsistent province mapping,Rebuilt province using city mapping
3,Missing gender,Filled with Unknown
4,Inconsistent category,Trimmed and standardized capitalization
5,Missing payment method,Filled with Unknown
6,Incorrect date type,Converted to datetime
7,Price outliers,Investigated and retained valid values


In [137]:
customers_clean.to_csv(
    "../data/processed/customers_clean.csv",
    index=False
)

In [138]:
transactions_clean.to_csv(
    "../data/processed/transactions_clean.csv",
    index=False
)

# Data Cleaning Summary

Proses Data Cleaning dilakukan berdasarkan masalah yang ditemukan
pada tahap Data Understanding.

## Cleaning yang dilakukan

1. Mengubah kolom tanggal menjadi tipe `datetime`.
2. Menghapus exact duplicate pada data transaksi.
3. Memvalidasi uniqueness `customer_id`.
4. Memvalidasi uniqueness `transaction_id`.
5. Menstandarkan nilai `city`.
6. Memperbaiki hubungan `city` dan `province`.
7. Menangani missing `gender` menggunakan kategori `Unknown`.
8. Menstandarkan kategori transaksi.
9. Menangani missing `payment_method`.
10. Memvalidasi quantity, unit price, discount, dan revenue.
11. Memvalidasi formula revenue.
12. Menginvestigasi outlier pada harga.
13. Memvalidasi relationship customer dan transaction.
14. Melakukan final data quality validation.

## Duplicate Handling

Exact duplicate transaction dihapus karena setiap
`transaction_id` seharusnya merepresentasikan satu transaksi unik.

## Missing Value Handling

Missing value pada field non-critical seperti gender dan payment
method dipertahankan secara informasional menggunakan kategori
`Unknown`, sehingga transaksi/customer tidak perlu dihapus.

## Outlier Handling

Outlier harga tidak otomatis dihapus.

Nilai ekstrem tetap dipertahankan apabila masih memenuhi business
rules karena perbedaan kategori produk dapat menghasilkan rentang
harga yang berbeda secara signifikan.

## Validation Result

Final validation dilakukan terhadap:

- primary key uniqueness
- missing critical fields
- numerical business rules
- revenue calculation
- referential integrity

Dataset hasil cleaning disimpan pada:

`data/processed/`

dan akan digunakan pada tahap berikutnya:

`03_rfm_analysis.ipynb`